In [ ]:
# --- 1. LIBRERÍAS Y CONFIGURACIÓN ---
import openpyxl
import pandas as pd
import plotly.express as px
import plotly.io as pio
import os

# Configuración de renderizado (para que se vea en navegador si falla en notebook)
pio.renderers.default = "notebook" 

In [3]:
# --- CONFIGURACIÓN DE COLORES ---
COLORES = {
    'Verde_Fuerte': '#74b404', # Pagado / Bueno / Con Fiador
    'Verde_Claro':  '#cdfc7d', 
    'Rojo_Corp':    '#aa044c', # Impago / Riesgo / Sin Fiador
    'Morado_Os':    '#876784',
    'Morado_Cl':    '#b09eae'
}

# Mapa para Pagado/Impago
MAPA_COLORES = {'Pagado': COLORES['Verde_Fuerte'], 'Impago': COLORES['Rojo_Corp']}

In [11]:
df = pd.read_excel(os.path.join('..', 'Datos', 'Originales', 'información_préstamos.xlsx'))

# 3. FILTRO GLOBAL: Solo Vivienda
df_viv = df[df['Proposito'].astype(str).str.contains('Vivienda', case=False, na=False)].copy()

# 4. Etiquetas básicas
df_viv['Impago_Label'] = df_viv['Impago'].map({0: 'Pagado', 1: 'Impago'})

print(f"Dataset listo. Trabajando exclusivamente con {len(df_viv)} casos de Vivienda.")

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [4]:
print(df['Proposito'].unique()) #El nuestro es vivienda

NameError: name 'df' is not defined

In [4]:
df.head(5)

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima
0,S97R7X,18,61628,83011,397,113,1,8.06,48,0.45,Doctorado,Autónomo,Casado,1,0,Automóvil,0,0,155.80
1,T3ZE0N,69,19485,25474,784,46,2,15.04,48,0.15,Grado Universitario,Tiempo parcial,Soltero,0,1,Educación,0,0,24.20
2,RLGTBY,50,82410,68642,486,14,3,21.96,12,0.71,Escolar,Tiempo parcial,Divorciado,1,0,Automóvil,1,1,58.33
3,BZ86CV,64,132974,208339,308,10,1,24.26,12,0.61,Máster,Desempleado,Casado,1,0,Negocios,1,0,284.46
4,5OD75M,62,51411,113847,412,47,2,5.73,36,0.70,Grado Universitario,Autónomo,Casado,0,0,Negocios,0,0,101.77


In [ ]:
#GRAFICO 1
fig1 = px.violin(
    df_viv, 
    y="Ratio_Deuda_Ingresos", 
    x="Impago_Label", 
    color="Impago_Label",
    box=True,               
    points="all",            
    hover_data=['Ingresos', 'Scoring_Crediticio'], 

    color_discrete_map={
        'Pagado': COLORES['Verde_Fuerte'], 
        'Impago': COLORES['Rojo_Corp']
    }, 
    title="<b>Distribución del Endeudamiento (Ratio Deuda/Ingresos)</b><br><sup>Préstamos de Vivienda: Clientes al Corriente vs. En Mora</sup>"
)


fig1.update_layout(
    xaxis_title="Estado del Préstamo",
    yaxis_title="Ratio Deuda / Ingresos (%)",
    legend_title="Estado",
    template="plotly_white",
    font=dict(size=12),
    title_font_size=16
)

fig1.show()
fig1.write_html("1_Perfil_riesgo.html") 
fig1.write_image("1_Perfil_riesgo.png")

In [ ]:
#GRAFICO 2
fig2 = px.scatter(
    df_viv, 
    x="Ingresos", 
    y="Monto_Inicial",
    color="Impago_Label", 
    facet_col="Impago_Label",    
    size="Ratio_Deuda_Ingresos", 
    size_max=10, 
    opacity=0.3,                 
    hover_data=['Scoring_Crediticio'],
    
    color_discrete_map={
        'Pagado': COLORES['Verde_Fuerte'], 
        'Impago': COLORES['Rojo_Corp']
    },
    title="<b>Mapa de Riesgo Comparativo: Ingresos vs. Monto Inicial</b><br><sup>Segmentación de Vivienda: Análisis de densidad y concentración de impagos</sup>"
)


fig2.update_layout(
    template="plotly_white",
    xaxis_title="Ingresos Anuales",
    yaxis_title="Monto del Préstamo",
    legend_title="Estado"
)


fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig2.show()
fig2.write_html("2_Ingresos_monto.html") 
fig2.write_image("2_Ingresos_monto.png")


In [98]:
#GRAFICO 3
variables = ['Estado_Civil', 'Tipo_Jornada_Laboral', 'Estudios']
prefijos = ['3a', '3b', '3c'] # Para nombrar los archivos ordenadamente

for var, pre in zip(variables, prefijos):
    df_agrupado = df_viv.groupby([var, 'Impago_Label']).size().reset_index(name='Cantidad')
    df_agrupado['Porcentaje'] = df_agrupado.groupby(var)['Cantidad'].transform(lambda x: 100 * x / x.sum())
    
    fig3 = px.bar(
        df_agrupado,
        x=var,
        y='Porcentaje',
        color='Impago_Label',
        text=df_agrupado['Porcentaje'].apply(lambda x: '{0:1.2f}%'.format(x)),
        color_discrete_map={
            'Pagado': COLORES['Verde_Fuerte'], 
            'Impago': COLORES['Rojo_Corp']
        },
        title=f"<b>Riesgo de Impago por {var.replace('_', ' ')}</b>"
    )
    
    fig3.update_layout(
        yaxis_title="Porcentaje (%)",
        xaxis_title=var.replace('_', ' '),
        template="plotly_white",
        uniformtext_minsize=8, 
        uniformtext_mode='hide'
    )
    
    fig3.show()
    fig3.write_html(f"{pre}_{var}.html")
    fig3.write_image(f"{pre}_{var}.png")



In [99]:
# GRÁFICO 4:
col_fiador = 'Fiador' 

df_viv['Fiador_Label'] = df_viv[col_fiador].map({0: 'Sin Fiador', 1: 'Con Fiador'})

fig4 = px.histogram(
    df_viv,
    x="Ratio_Interes",  
    color="Fiador_Label",
    marginal="box",     
    nbins=30, 
    opacity=0.7, 
    barmode="overlay", 
    color_discrete_map={
        'Sin Fiador': COLORES['Rojo_Corp'], 
        'Con Fiador': COLORES['Verde_Fuerte']
    },
    title="<b>Distribución de Tipos de Interés en Vivienda</b><br><sup>Impacto del Fiador en el coste del préstamo</sup>"
)


fig4.update_layout(
    xaxis_title="Tasa de Interés (%)",
    yaxis_title="Frecuencia (Nº de Préstamos)",
    template="plotly_white",
    legend_title="¿Tiene Fiador?",
    font=dict(size=12)
)

fig4.show()
fig4.write_html("4_Distribucion_Interes.html")
fig4.write_image("4_Distribucion_Interes.png")

In [110]:
# GRÁFICO 5
fig5 = px.box(
    df_viv,
    x='Impago_Label',
    y='Prima',
    color='Impago_Label',
    title='<b>Análisis de la Prima de Seguro: ¿Está el riesgo bien tarificado?</b>',
    labels={
        'Prima': 'Costo de la Prima (€)',
        'Impago_Label': '¿Entró en mora?'
    },
    # Aplicamos tu paleta personalizada COLORES
    color_discrete_map={
        'Pagado': COLORES['Verde_Fuerte'], 
        'Impago': COLORES['Rojo_Corp']
    },
    template='plotly_white'
)

# 3. Ajustes de estilo
fig5.update_layout(
    xaxis_title="Estado Final del Préstamo",
    yaxis_title="Costo de la Prima (€)"
)

fig5.show()
fig5.write_html("5_Analisis_Prima.html")
fig5.write_image("5_Analisis_Prima.png")

In [103]:
df_viv['Duracion_Anios'].describe()

count    51286.000000
mean         3.001267
std          1.415364
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          5.000000
Name: Duracion_Anios, dtype: float64

In [106]:
# GRÁFICO 6: 
df_viv['Duracion_Anios'] = df_viv['Duracion'] / 12
bins_duracion = [0, 1, 2, 3, 4, 5]
labels_duracion = ['1 año', '2 años', '3 años', '4 años', '5 años']
df_viv['Rango_Duracion'] = pd.cut(df_viv['Duracion_Anios'], bins=bins_duracion, labels=labels_duracion)

df_tasa = df_viv.groupby('Rango_Duracion', observed=False)['Impago'].mean().reset_index()
df_tasa['Tasa_Impago_Pct'] = df_tasa['Impago'] * 100 

fig6 = px.bar(
    df_tasa,
    x='Rango_Duracion',
    y='Tasa_Impago_Pct',
    # Añadimos el texto que se mostrará
    text=df_tasa['Tasa_Impago_Pct'].apply(lambda x: f'{x:.1f}%'), 
    title='<b>Riesgo Crítico por Plazo: Tasa de Impago Anual</b>',
    labels={'Rango_Duracion': 'Plazo del Préstamo', 'Tasa_Impago_Pct': '% Tasa de Impago'},
    template='plotly_white',
    color='Tasa_Impago_Pct',
    color_continuous_scale=[[0, COLORES['Verde_Fuerte']], [1, COLORES['Rojo_Corp']]]
)

fig6.update_traces(
    textposition='outside', 
    cliponaxis=False        
)

fig6.update_layout(
    yaxis_title="Tasa de Morosidad (%)",
    xaxis_title="Años de Duración",
    coloraxis_showscale=False,
    yaxis_range=[0, df_tasa['Tasa_Impago_Pct'].max() * 1.15] 
)

fig6.show()
fig6.write_html("6_Duracion_Riesgo.html")
fig6.write_image("6_Duracion_Riesgo.png")

In [117]:
# GRÁFICO 7
bins_edad = [18, 25, 35, 45, 55, 65, 120]
labels_edad = ['18-25', '26-35', '36-45', '46-55', '56-65', '+65']

df_viv['Rango_Edad'] = pd.cut(df_viv['Edad'], bins=bins_edad, labels=labels_edad)

fig7 = px.box(
    df_viv,
    x='Rango_Edad',
    y='Meses_Empleo',
    color='Impago_Label',

    category_orders={"Rango_Edad": labels_edad}, 
    title='<b>Perfil de Estabilidad: Antigüedad Laboral por Edad</b><br><sup>Comparativa de meses en el empleo actual</sup>',
    labels={'Rango_Edad': 'Rango de Edad', 'Meses_Empleo': 'Meses de Antigüedad'},
    # Aplicamos tu paleta personalizada
    color_discrete_map={
        'Pagado': COLORES['Verde_Fuerte'], 
        'Impago': COLORES['Rojo_Corp']
    },
    template='plotly_white'
)

fig7.update_layout(
    xaxis_title="Grupos de Edad (Ordenados)",
    yaxis_title="Meses en el Empleo Actual",
    legend_title="Estado",
    boxmode='group' 
)

fig7.show()
fig7.write_html("7_Estabilidad_Laboral.html")
fig7.write_image("7_Estabilidad_Laboral.png")

In [118]:
# --- GRÁFICO 8

fig8 = px.histogram(
    df_viv, 
    x="Scoring_Crediticio", 
    color="Impago_Label", 
    barmode="overlay", 
    title="<b>La Prueba de Fuego: Distribución del Scoring</b><br><sup>¿Es fiable el sistema de puntuación actual?</sup>",
    labels={'Scoring_Crediticio': 'Puntuación Crediticia (0-1000)', 'count': 'Frecuencia'},
    # Aplicamos tu paleta corporativa
    color_discrete_map={
        'Pagado': COLORES['Verde_Fuerte'], 
        'Impago': COLORES['Rojo_Corp']
    },
    opacity=0.6,
    template='plotly_white'
)

fig8.update_layout(yaxis_title="Cantidad de Clientes")

fig8.show()
fig8.write_html("8_Scoring.html")
fig8.write_image("8_Scoring.png")

In [ ]:
# GRÁFICO 9
fig9 = px.histogram(
    df_viv, 
    x="Personas_Cargo", 
    color="Impago_Label", 
    barmode="group",
    title="<b>Impacto de la Carga Familiar</b><br><sup>¿Aumenta el riesgo al tener más dependientes?</sup>",
    labels={'Personas_Cargo': 'Nº de Personas a Cargo', 'count': 'Nº de Clientes'},
    color_discrete_map={
        'Pagado': COLORES['Verde_Fuerte'], 
        'Impago': COLORES['Rojo_Corp']
    },
    template='plotly_white'
)

fig9.update_layout(bargap=0.2)

fig9.update_traces(texttemplate='%{y}', textposition='outside')
fig9.update_layout(yaxis_title="Cantidad de Clientes")

fig9.show()
fig9.write_html("9_Carga_Familiar.html")
fig9.write_image("9_Carga_Familiar.png")

In [ ]:
# GRÁFICO 10: 
fig10 = px.histogram(
    df_viv, 
    x="Num_Creditos", 
    color="Impago_Label", 
    barmode="group",
    title="<b>El Riesgo del Multi-Crédito</b><br><sup>Relación entre número de préstamos activos y morosidad</sup>",
    labels={'Num_Creditos': 'Número de Créditos Vigentes', 'count': 'Frecuencia'},
    color_discrete_map={
        'Pagado': COLORES['Verde_Fuerte'], 
        'Impago': COLORES['Rojo_Corp']
    },
    template='plotly_white'
)

# Etiquetas numéricas para ver claro el dato
fig10.update_traces(texttemplate='%{y}', textposition='outside')
fig10.update_layout(yaxis_title="Cantidad de Clientes")

fig10.show()
fig10.write_html("10_Multi_Credito.html")
fig10.write_image("10_Multi_Credito.png")

**---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------**


**----------------------------------------------------------------------------------------------------------------------------------**
**GRAFICOS BONITOS PARA INFORME**
**----------------------------------------------------------------------------------------------------------------------------------**

In [133]:
# --- GRÁFICO 12:

df_viv['Tiene_Hipoteca'] = df_viv['Posesion_Hipoteca'].map({0: 'No tiene', 1: 'Sí tiene'})
df_viv['Impago_Label'] = df_viv['Impago'].map({0: 'Pagado', 1: 'Impago'})

fig11 = px.parallel_categories(
    df_viv,
    dimensions=['Estudios', 'Tiene_Hipoteca', 'Impago_Label'], 
    color="Impago",
    
   
    color_continuous_scale=[[0, COLORES['Verde_Fuerte']], [1, COLORES['Rojo_Corp']]],
    
    title="<b>Flujo de Riesgo: Educación e Historial Hipotecario</b><br><sup>¿Protege tener experiencia previa con hipotecas?</sup>"
)

fig11.update_layout(
    template="plotly_white",
    margin=dict(t=80, l=50, r=50, b=20),
    coloraxis_showscale=False 
)

fig11.show()
fig11.write_html("12_Flujo_Riesgo.html")
fig11.write_image("12_Sunburst_Perfil.png")

In [136]:
# --- GRÁFICO 3: PERFIL DEMOGRÁFICO COMBINADO (CON %) ---
df_sunburst = df_viv.groupby(['Estado_Civil', 'Tipo_Jornada_Laboral', 'Impago_Label']).size().reset_index(name='Cantidad')

fig3_5 = px.sunburst(
    df_sunburst,
    path=['Estado_Civil', 'Tipo_Jornada_Laboral', 'Impago_Label'], 
    values='Cantidad', 
    color='Impago_Label', 
    
    color_discrete_map={
        'Pagado': COLORES['Verde_Fuerte'], 
        'Impago': COLORES['Rojo_Corp'],
        '(?)': '#bdc3c7'
    },
    
    title="<b>3. Perfil Demográfico del Riesgo</b><br><sup>Jerarquía: Estado Civil > Jornada Laboral > Resultado</sup>"
)

fig3_5.update_traces(
    textinfo='label+percent entry',
    insidetextorientation='radial' 
)

fig3_5.update_layout(
    template="plotly_white",
    font=dict(size=14),
    margin=dict(t=60, l=10, r=10, b=10)
)

fig3_5.show()
fig3_5.write_html("3.5_Sunburst_Perfil.html")
fig3_5.write_image("3.5_Sunburst_Perfil.png")